# FoodBridge SG — Beneficiary Dashboard Engine
### IS215 Digital Business – Technologies and Transformation

This notebook powers the **beneficiary-facing dashboard** of FoodBridge SG.  
It reads raw historical sales data, computes actual observed daily surplus at each store using a rolling-average proxy, then runs the AI matching engine to recommend the best stores for each NPO to collect from.

**Inputs:** Raw CSVs (`training.csv`, `items.csv`, `stores.csv`, `transactions.csv`, `oil.csv`, `holidays_events.csv`)  
**Outputs:** `dashboard_payload.json` — a single file containing all dashboard data  

> **Run order:** This notebook is self-contained. `analytics.ipynb` does not need to be run first — the beneficiary dashboard computes actual surplus directly from raw data rather than relying on the RF forecast.

---
## Configuration & Imports

All tunable parameters are defined here in one place — making it easy to adjust behaviour without hunting through the code:

- `MAX_DISTANCE_KM` — radius within which stores are considered "nearby" for a given NPO
- `TOP_N_MATCHES` — maximum number of AI match cards shown on the dashboard
- `FOOD_PRICE_PER_KG` — estimated SGD value per kg used to compute the "money saved" metric

The dashboard reads raw sales data directly and computes actual observed surplus — no pre-generated forecast file is needed.  
The **Haversine formula** (imported via `math`) calculates real-world distance between two GPS coordinates — used in the matching engine.

In [ ]:
import pandas as pd
import numpy as np
import json
from math import radians, sin, cos, sqrt, atan2
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Tunable parameters ────────────────────────────────────────────────────
MAX_DISTANCE_KM   = 10.0   # Stores beyond this are excluded from matching
TOP_N_MATCHES     = 5      # Max match cards shown per beneficiary
FOOD_PRICE_PER_KG = 4.50   # SGD per kg — used for money saved calculation
TODAY             = datetime.today().strftime('%d %b %Y')

print("=" * 60)
print("  FoodBridge SG — Beneficiary Dashboard Engine")
print(f"  Running for: {TODAY}")
print("=" * 60)

---
## Stage 1: Load All Raw Data

We load the same 6 source CSVs used by the analytics pipeline (no `testing.csv` needed here — we are generating outputs, not predicting future rows).

All `date` columns are converted to `datetime` immediately.  
This is a defensive practice — downstream stages compare dates using `>=` and `-`, which will raise a `TypeError` if dates are still strings.

In [ ]:
print("\n[1/7] Loading raw data...")

train        = pd.read_csv('training.csv')
items        = pd.read_csv('items.csv')
stores       = pd.read_csv('stores.csv')
transactions = pd.read_csv('transactions.csv')
oil          = pd.read_csv('oil.csv')
holidays     = pd.read_csv('holidays_events.csv')

for df, col in [(train,'date'),(transactions,'date'),(oil,'date'),(holidays,'date')]:
    df[col] = pd.to_datetime(df[col])

print(f"  train: {len(train):,} rows | items: {len(items)} | stores: {len(stores)}")
print("  ✓ Raw data loaded")

---
## Stage 2: Compute Actual Daily Surplus

Rather than relying on ML-predicted surplus from `analytics.ipynb`, the beneficiary dashboard computes **actual observed surplus** directly from raw historical data.

**Why use actual data here instead of RF predictions?**  
Beneficiaries need to act on real, available surplus — not a forecast. The rolling-average proxy gives us the best available estimate of what is *actually left unsold* at each store on the latest recorded day, making the matching recommendations grounded in observed reality rather than a model's extrapolation.

**The computation:**
1. Filter to perishable items only (same as analytics)
2. Compute a 7-day shifted rolling average of sales per store-item pair — this represents the "expected" sales volume
3. `estimated_waste = (avg_sales_7d − unit_sales).clip(lower=0)` — the gap between expectation and reality, floored at zero
4. Take the **most recent date** in the dataset and aggregate `estimated_waste` to store level

The resulting `surplus` DataFrame has the same column names (`store_nbr`, `total_predicted_waste`, `top_waste_category`, `city`, `state`, `type`) that the AI Matching Engine in Stage 5 expects — so all downstream cells work without modification.

In [ ]:
print("\n[2/7] Computing actual daily surplus from historical data...")

# Filter to perishable items only
perishables = items[items['perishable'] == 1][['item_nbr', 'family']]
df = train.merge(perishables, on='item_nbr', how='inner')
df = df.sort_values(['store_nbr', 'item_nbr', 'date']).reset_index(drop=True)

# 7-day shifted rolling average — "expected" sales, using prior days only (no leakage)
df['avg_sales_7d'] = (
    df.groupby(['store_nbr', 'item_nbr'])['unit_sales']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

# Actual surplus proxy: gap between expected and actual, never negative
df['estimated_waste'] = (df['avg_sales_7d'] - df['unit_sales']).clip(lower=0)
df = df.dropna(subset=['avg_sales_7d'])

# Use the most recent date in the dataset
latest_date = df['date'].max()
print(f"  Using actual data for: {latest_date.date()}")

latest_df = df[df['date'] == latest_date].copy()

# Aggregate to store level
actual_surplus = (
    latest_df.groupby('store_nbr')
    .agg(
        total_surplus=('estimated_waste', 'sum'),
        top_waste_category=('family', lambda x: latest_df.loc[x.index]
                            .groupby('family')['estimated_waste']
                            .sum().idxmax())
    )
    .reset_index()
)

actual_surplus = actual_surplus.merge(
    stores[['store_nbr', 'city', 'state', 'type']], on='store_nbr', how='left'
)
actual_surplus['forecast_date'] = latest_date.date()

# Rename to match the column name the matching engine expects
surplus = actual_surplus.rename(columns={'total_surplus': 'total_predicted_waste'})

print(f"  ✓ Actual surplus computed — {len(surplus)} stores with surplus data")
surplus.head()

---
## Stage 3: Beneficiary Profile & Pickup History

In production, this data comes from the app's database — each NPO's registration record and their complete history of confirmed pickups.

For this prototype, we simulate it directly in code.

**Key design decisions:**
- `pickup_date` is converted to `datetime` **immediately** after the DataFrame is created — before any subset (`completed`/`pending`) is formed. This is critical: if conversion happens later, date comparisons in Stage 6 raise a `TypeError` because strings and `datetime` objects cannot be compared with `>=`.
- `history_counts` aggregates past completed collections per store — this feeds into the matching score as the "relationship history" signal.

In [ ]:
print("\n[3/7] Loading beneficiary profile and history...")

BENEFICIARY = {
    'id':          'BEN_001',
    'name':        'Food From The Heart',
    'latitude':    1.3521,
    'longitude':   103.8198,
    'preferences': ['BREAD/BAKERY', 'DELI', 'DAIRY EGGS', 'PRODUCE'],
    'joined_date': '2024-01-15'
}

pickup_history = pd.DataFrame([
    {'item_name': 'Vegetables', 'quantity_kg': 30, 'store_name': 'FairPrice Xtra',
     'store_nbr': 3, 'distance_km': 2.3,
     'pickup_date': (datetime.today() - timedelta(days=2)).strftime('%Y-%m-%d'), 'status': 'pending'},
    {'item_name': 'Condiments',  'quantity_kg': 20, 'store_name': 'FairPrice Xtra',
     'store_nbr': 3, 'distance_km': 2.3,
     'pickup_date': (datetime.today() - timedelta(days=3)).strftime('%Y-%m-%d'), 'status': 'complete'},
    {'item_name': 'Rice',        'quantity_kg': 40, 'store_name': 'FairPrice Xtra',
     'store_nbr': 3, 'distance_km': 2.3,
     'pickup_date': (datetime.today() - timedelta(days=6)).strftime('%Y-%m-%d'), 'status': 'complete'},
    {'item_name': 'Bread',       'quantity_kg': 25, 'store_name': 'Cold Storage',
     'store_nbr': 7, 'distance_km': 2.7,
     'pickup_date': (datetime.today() - timedelta(days=10)).strftime('%Y-%m-%d'), 'status': 'complete'},
    {'item_name': 'Dairy',       'quantity_kg': 15, 'store_name': 'NTUC FairPrice',
     'store_nbr': 11, 'distance_km': 3.4,
     'pickup_date': (datetime.today() - timedelta(days=14)).strftime('%Y-%m-%d'), 'status': 'complete'},
    {'item_name': 'Bread',       'quantity_kg': 35, 'store_name': 'FairPrice Xtra',
     'store_nbr': 3, 'distance_km': 2.3,
     'pickup_date': (datetime.today() - timedelta(days=20)).strftime('%Y-%m-%d'), 'status': 'complete'},
])

# Convert to datetime BEFORE creating any subsets — prevents TypeError in Stage 6
pickup_history['pickup_date'] = pd.to_datetime(pickup_history['pickup_date'])

history_counts = (
    pickup_history[pickup_history['status'] == 'complete']
    .groupby('store_nbr')['item_name']
    .count()
    .reset_index()
    .rename(columns={'item_name': 'collections'})
)

print(f"  ✓ Profile loaded — {len(pickup_history)} past pickups")
pickup_history

---
## Stage 4: Store Coordinates

The matching engine needs GPS coordinates to calculate distance between each NPO and each store.  
The `stores.csv` only has city names, so we map them to approximate latitude/longitude values.

**Haversine formula:**  
We use the Haversine formula rather than straight-line (Euclidean) distance because it accounts for the curvature of the Earth — giving accurate real-world distances in kilometres between two GPS points.

> In a production system, replace this dictionary with a call to the **Google Maps Geocoding API** to get precise coordinates for each store address.

In [ ]:
city_coords = {
    'Quito':         (1.3521,  103.8198),
    'Guayaquil':     (1.3000,  103.8000),
    'Cuenca':        (1.2800,  103.8300),
    'Ambato':        (1.3200,  103.7900),
    'Latacunga':     (1.3700,  103.8500),
    'Riobamba':      (1.3400,  103.8100),
    'Ibarra':        (1.4000,  103.8700),
    'Salinas':       (1.4200,  103.8000),
    'Daule':         (1.3100,  103.7800),
    'Santo Domingo': (1.3600,  103.8200),
    'Cayambe':       (1.3900,  103.8400),
    'Manta':         (1.3300,  103.7700),
}

def haversine_km(lat1, lon1, lat2, lon2):
    """Real-world distance in km between two GPS coordinates."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coords(city):
    return city_coords.get(city, (1.3521, 103.8198))

stores['latitude']  = stores['city'].apply(lambda c: get_coords(c)[0])
stores['longitude'] = stores['city'].apply(lambda c: get_coords(c)[1])

print("[4/7] Store coordinates mapped")
stores[['store_nbr','city','latitude','longitude']].head()

---
## Stage 5: AI Matching Engine

This is the core AI component of the dashboard.  
For each store with **actual observed surplus**, we compute a **match score (0–100)** for the beneficiary.

**Why actual data instead of forecasts?**  
The matching engine runs on the current day's real surplus — this means the NPO is always matched with stores that have food *available now*, not just predicted to have food. This makes recommendations immediately actionable.

**Scoring formula:**

| Signal | Weight | Logic |
|---|---|---|
| Surplus amount | 35% | More food available = higher score |
| Distance | 45% | Closer store = higher score (most important for operational efficiency) |
| Past pickups | 20% | Stores the NPO has collected from before score higher (relationship signal) |
| Category bonus | +10% flat | If the store's top surplus item matches the NPO's food preferences |

**Why distance is weighted most heavily (45%)?**  
For an NPO with limited transport resources, distance is the single biggest operational constraint. A store with slightly less food but much closer is almost always better than a distant high-surplus store.

**Deduplication fix:**  
`surplus` already contains `city`, `state`, and `type` from Stage 2. Merging with `stores` again without dropping these first would create `city_x`/`city_y` duplicate columns — causing a `KeyError: 'city'`. We explicitly drop them before the merge.

In [ ]:
print("\n[5/7] Running AI matching engine...")

WEIGHT_SURPLUS  = 0.35
WEIGHT_DISTANCE = 0.45
WEIGHT_HISTORY  = 0.20

def compute_match_score(surplus_units, distance_km, history_count, top_category, preferences):
    surplus_score  = min(surplus_units / 100.0, 1.0)
    distance_score = max(0, 1.0 - (distance_km / MAX_DISTANCE_KM))
    history_score  = min(history_count / 10.0, 1.0)
    category_bonus = 0.10 if str(top_category).upper() in [p.upper() for p in preferences] else 0.0
    raw = (WEIGHT_SURPLUS  * surplus_score +
           WEIGHT_DISTANCE * distance_score +
           WEIGHT_HISTORY  * history_score +
           category_bonus)
    return round(min(raw, 1.0) * 100, 1)

# Drop overlapping columns before merge to prevent _x/_y column name conflicts
cols_to_drop = [c for c in ['city','state','type'] if c in surplus.columns]
surplus_clean = surplus.drop(columns=cols_to_drop)
surplus_with_loc = surplus_clean.merge(
    stores[['store_nbr','city','state','type','latitude','longitude']],
    on='store_nbr', how='left'
)

matches, nearby = [], []
ben_lat, ben_lon = BENEFICIARY['latitude'], BENEFICIARY['longitude']
prefs = BENEFICIARY['preferences']

for _, store in surplus_with_loc.iterrows():
    if pd.isna(store['latitude']):
        continue
    dist = haversine_km(ben_lat, ben_lon, store['latitude'], store['longitude'])
    if dist > MAX_DISTANCE_KM:
        continue

    hist_row = history_counts[history_counts['store_nbr'] == store['store_nbr']]
    hist_cnt = int(hist_row['collections'].values[0]) if len(hist_row) > 0 else 0

    score = compute_match_score(
        surplus_units = store['total_predicted_waste'],
        distance_km   = dist,
        history_count = hist_cnt,
        top_category  = store.get('top_waste_category', ''),
        preferences   = prefs
    )

    record = {
        'store_nbr':       int(store['store_nbr']),
        'store_name':      f"{store['city']} Store #{int(store['store_nbr'])}",
        'city':            store['city'],
        'store_type':      store['type'],
        'distance_km':     round(dist, 1),
        'predicted_waste': round(store['total_predicted_waste'], 0),
        'top_category':    store.get('top_waste_category', 'N/A'),
        'past_pickups':    hist_cnt,
        'match_score':     score,
        'surplus_status':  ('High surplus'     if store['total_predicted_waste'] >= 30 else
                            'Moderate surplus' if store['total_predicted_waste'] >= 10 else
                            'Low surplus'),
        'is_matched':      score >= 60
    }
    nearby.append(record)
    if score >= 60:
        matches.append(record)

matches.sort(key=lambda x: x['match_score'], reverse=True)
nearby.sort(key=lambda x: x['predicted_waste'], reverse=True)
matches = matches[:TOP_N_MATCHES]

print(f"  ✓ {len(matches)} AI matches generated | {len(nearby)} nearby stores found")
pd.DataFrame(matches)[['store_name','distance_km','predicted_waste','match_score','surplus_status']]

---
## Stage 6: Compute Dashboard Metrics

This stage calculates the four summary numbers shown at the top of the beneficiary dashboard:

**Total received (kg):** Sum of all completed pickup quantities — reflects the NPO's cumulative impact.

**Money saved:** Total kg × `FOOD_PRICE_PER_KG`. This translates food weight into a dollar value the NPO can use in grant applications and reports to donors.

**Month-over-month % change:** Compares the last 30 days against the prior 30 days.  
- `cutoff_30` = 30 days ago  
- `cutoff_60` = 60 days ago  
- Recent period = cutoff_30 → today  
- Prior period = cutoff_60 → cutoff_30  
This requires `pickup_date` to already be `datetime` — which is why the conversion was done in Stage 3.

**Top category:** Most collected food type by weight — drives the "top category" metric card and influences which stores are ranked higher in matches.

In [ ]:
print("\n[6/7] Computing dashboard metrics...")

completed = pickup_history[pickup_history['status'] == 'complete']
pending   = pickup_history[pickup_history['status'] == 'pending']

total_received_kg = int(completed['quantity_kg'].sum())
money_saved       = round(total_received_kg * FOOD_PRICE_PER_KG, 2)

cutoff_30 = datetime.today() - timedelta(days=30)
cutoff_60 = datetime.today() - timedelta(days=60)

recent_kg = completed[completed['pickup_date'] >= cutoff_30]['quantity_kg'].sum()
prior_kg  = completed[
    (completed['pickup_date'] >= cutoff_60) &
    (completed['pickup_date'] <  cutoff_30)
]['quantity_kg'].sum()

pct_change_received = round((recent_kg - prior_kg) / prior_kg * 100, 1) if prior_kg > 0 else 0
pct_change_saved    = pct_change_received   # derived from same weight

category_counts = completed.groupby('item_name')['quantity_kg'].sum().sort_values(ascending=False)
top_category    = category_counts.index[0] if len(category_counts) > 0 else prefs[0]

metrics = {
    'total_received_kg':    total_received_kg,
    'total_received_label': f"{total_received_kg} kg",
    'pct_change_received':  pct_change_received,
    'money_saved':          money_saved,
    'money_saved_label':    f"${money_saved:,.0f}",
    'pct_change_saved':     pct_change_saved,
    'ai_matches_today':     len(matches),
    'pending_pickups':      len(pending),
    'top_category':         top_category,
    'last_updated':         datetime.now().strftime('%a %d %b %Y, %I:%M %p'),
    'beneficiary_name':     BENEFICIARY['name'],
}

print(f"  Total received : {metrics['total_received_label']}  ({'+' if pct_change_received>=0 else ''}{pct_change_received}% MoM)")
print(f"  Money saved    : {metrics['money_saved_label']}  ({'+' if pct_change_saved>=0 else ''}{pct_change_saved}% MoM)")
print(f"  AI matches     : {len(matches)}")
print(f"  Top category   : {top_category}")
metrics

---
## Stage 7: Top Categories Breakdown & Recently Received

**Top categories:**  
Aggregates completed pickups by food type and normalises to a 0–100% scale.  
The percentage is relative to the most-collected category (max = 100%) — this powers the horizontal bar chart on the dashboard and helps NPOs understand their own food collection patterns.

**Recently received:**  
Takes the 5 most recent pickups and formats them for the history section of the dashboard.  
`delta` is the number of days since pickup, converted to a human-readable label ("Today", "Yesterday", "3 days ago").

In [ ]:
cat_totals = completed.groupby('item_name')['quantity_kg'].sum().sort_values(ascending=False)
max_qty    = cat_totals.max() if len(cat_totals) > 0 else 1

top_categories = [
    {'category': cat, 'quantity_kg': int(qty), 'percentage': round(qty / max_qty * 100, 0)}
    for cat, qty in cat_totals.head(5).items()
]

recently_received = []
for _, row in pickup_history.sort_values('pickup_date', ascending=False).head(5).iterrows():
    delta = (datetime.today() - pd.to_datetime(row['pickup_date'])).days
    days_label = 'Today' if delta == 0 else 'Yesterday' if delta == 1 else f"{delta} days ago"
    recently_received.append({
        'item_name':   row['item_name'],
        'quantity_kg': int(row['quantity_kg']),
        'store_name':  row['store_name'],
        'distance_km': row['distance_km'],
        'days_ago':    days_label,
        'status':      row['status'],
        'pickup_date': str(row['pickup_date'])[:10]
    })

print("Top categories:")
for c in top_categories:
    print(f"  {c['category']:15s}  {c['quantity_kg']} kg  ({c['percentage']}%)")

print("\nRecently received:")
for r in recently_received:
    print(f"  {r['item_name']:12s}  {r['quantity_kg']} kg  {r['days_ago']:15s}  [{r['status']}]")

---
## Stage 8: Write All JSON Output Files

We save each dashboard section as an individual JSON file, plus one combined `dashboard_payload.json` that contains everything.

**Why JSON?**  
JSON is the universal format for web APIs. Your frontend (React, Flutter, HTML prototype) can fetch `dashboard_payload.json` on load and map each key directly to its corresponding UI section — no transformation needed.

**File map:**

| File | Dashboard section |
|---|---|
| `dashboard_metrics.json` | 4 metric cards at the top |
| `ai_matches.json` | AI-recommended match cards |
| `top_categories.json` | Category bar chart |
| `nearby_stores.json` | All stores near you table |
| `recently_received.json` | Pickup history rows |
| `dashboard_payload.json` | All of the above in one file |

In [ ]:
print("\n[7/7] Writing dashboard JSON files...")

def save_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2, default=str)
    print(f"  ✓ Saved: {filename}")

save_json(metrics,           'dashboard_metrics.json')
save_json(matches,           'ai_matches.json')
save_json(top_categories,    'top_categories.json')
save_json(nearby,            'nearby_stores.json')
save_json(recently_received, 'recently_received.json')

combined_payload = {
    'generated_at':      datetime.now().isoformat(),
    'metrics':           metrics,
    'ai_matches':        matches,
    'top_categories':    top_categories,
    'nearby_stores':     nearby,
    'recently_received': recently_received,
}
save_json(combined_payload, 'dashboard_payload.json')

print("\n" + "=" * 60)
print("  BENEFICIARY DASHBOARD ENGINE COMPLETE")
print("=" * 60)
print("  dashboard_payload.json is ready for your frontend.")
print("  Matches are based on actual observed surplus from the")
print("  latest date in the dataset — no forecast needed.")
print("  Re-run this notebook daily at 6 AM to refresh the data.")
print("=" * 60)